## Imports

In [1]:
import os
import pandas as pd
from tqdm import tqdm

DATASET_DIR = "/kaggle/input/datasets/tunguz/xview2-challenge-dataset-tier-3-data/tier3"
OUTPUT_CSV = "tier3_pairs.csv"

## Helper function: To find files

In [2]:
def find_all_files(root_dir, ext):
    all_files = []
    for root, _, files in os.walk(root_dir):
        for f in files:
            if f.lower().endswith(ext):
                all_files.append(os.path.join(root, f))
    return all_files

## Collect Pre/Post Images

In [3]:
png_files = find_all_files(DATASET_DIR, ".png")
json_files = find_all_files(DATASET_DIR, ".json")

In [4]:
pre_images = sorted([p for p in png_files if "pre_disaster" in p])
post_images = sorted([p for p in png_files if "post_disaster" in p])

In [5]:
print("Pre images:", len(pre_images))
print("Post images:", len(post_images))

Pre images: 6369
Post images: 6369


## Building pair dataset

In [6]:
pairs = []

for post_path in tqdm(post_images):

    pre_path = post_path.replace(
        "post_disaster",
        "pre_disaster"
    )

    if not os.path.exists(pre_path):
        continue

    # Build exact post-disaster JSON path
    json_path = (
        post_path
        .replace("/images/", "/labels/")
        .replace(".png", ".json")
    )

    if not os.path.exists(json_path):
        continue

    pairs.append({
        "pre_image": pre_path,
        "post_image": post_path,
        "label_json": json_path
    })

100%|██████████| 6369/6369 [00:02<00:00, 2425.77it/s]


In [7]:
df = pd.DataFrame(pairs)
print("Final pairs found:", len(df))
df.head()

Final pairs found: 6369


,pre_image,post_image,label_json
0,/kaggle/input/datasets/tunguz/xview2-challenge...,/kaggle/input/datasets/tunguz/xview2-challenge...,/kaggle/input/datasets/tunguz/xview2-challenge...
1,/kaggle/input/datasets/tunguz/xview2-challenge...,/kaggle/input/datasets/tunguz/xview2-challenge...,/kaggle/input/datasets/tunguz/xview2-challenge...
2,/kaggle/input/datasets/tunguz/xview2-challenge...,/kaggle/input/datasets/tunguz/xview2-challenge...,/kaggle/input/datasets/tunguz/xview2-challenge...
3,/kaggle/input/datasets/tunguz/xview2-challenge...,/kaggle/input/datasets/tunguz/xview2-challenge...,/kaggle/input/datasets/tunguz/xview2-challenge...
4,/kaggle/input/datasets/tunguz/xview2-challenge...,/kaggle/input/datasets/tunguz/xview2-challenge...,/kaggle/input/datasets/tunguz/xview2-challenge...


## Verify Annotation File Pairing

This section validates that every image pair is linked to a post-disaster annotation file and confirms that no pre-disaster label files are included in the final dataset.

In [8]:
print("Total pairs:", len(df))

print(
    "Pre labels:",
    df["label_json"].str.contains("pre_disaster").sum()
)

print(
    "Post labels:",
    df["label_json"].str.contains("post_disaster").sum()
)

Total pairs: 6369
Pre labels: 0
Post labels: 6369


## Random Sample Inspection of Image–Label Pairs

This section randomly samples a few records from the dataset to manually verify that each pre-disaster image, post-disaster image, and post-disaster annotation file are correctly matched.

In [9]:
import random

for _ in range(5):
    row = df.sample(1).iloc[0]

    print()
    print("PRE :", os.path.basename(row["pre_image"]))
    print("POST:", os.path.basename(row["post_image"]))
    print("JSON:", os.path.basename(row["label_json"]))


PRE : portugal-wildfire_00000527_pre_disaster.png
POST: portugal-wildfire_00000527_post_disaster.png
JSON: portugal-wildfire_00000527_post_disaster.json

PRE : portugal-wildfire_00000514_pre_disaster.png
POST: portugal-wildfire_00000514_post_disaster.png
JSON: portugal-wildfire_00000514_post_disaster.json

PRE : woolsey-fire_00000627_pre_disaster.png
POST: woolsey-fire_00000627_post_disaster.png
JSON: woolsey-fire_00000627_post_disaster.json

PRE : portugal-wildfire_00000693_pre_disaster.png
POST: portugal-wildfire_00000693_post_disaster.png
JSON: portugal-wildfire_00000693_post_disaster.json

PRE : portugal-wildfire_00000244_pre_disaster.png
POST: portugal-wildfire_00000244_post_disaster.png
JSON: portugal-wildfire_00000244_post_disaster.json


## Inspect xBD Annotation Structure

This section examines the structure of a randomly selected annotation file to verify the presence of required fields and ensure compatibility with the mask-generation pipeline.

In [10]:
import json

row = df.sample(1).iloc[0]

with open(row["label_json"]) as f:
    data = json.load(f)

print(data["features"].keys())

dict_keys(['lng_lat', 'xy'])


In [11]:
print(row["label_json"])

/kaggle/input/datasets/tunguz/xview2-challenge-dataset-tier-3-data/tier3/labels/portugal-wildfire_00000855_post_disaster.json


## Validate Building Annotation Availability

This section analyzes multiple annotation files to determine how many contain building polygons and how many correspond to empty or unlabeled image tiles.

In [12]:
count_nonzero = 0

for _ in range(20):
    row = df.sample(1).iloc[0]

    with open(row["label_json"], "r") as f:
        data = json.load(f)

    n = len(data["features"]["xy"])

    print(
        os.path.basename(row["label_json"]),
        " -> ",
        n
    )

    if n > 0:
        count_nonzero += 1

print("\nSamples with features:", count_nonzero)

portugal-wildfire_00001483_post_disaster.json  ->  0
pinery-bushfire_00001084_post_disaster.json  ->  0
portugal-wildfire_00000480_post_disaster.json  ->  0
pinery-bushfire_00001296_post_disaster.json  ->  0
portugal-wildfire_00000496_post_disaster.json  ->  16
moore-tornado_00000125_post_disaster.json  ->  146
tuscaloosa-tornado_00000153_post_disaster.json  ->  0
portugal-wildfire_00000590_post_disaster.json  ->  0
pinery-bushfire_00000240_post_disaster.json  ->  0
portugal-wildfire_00000808_post_disaster.json  ->  21
nepal-flooding_00000017_post_disaster.json  ->  32
portugal-wildfire_00000303_post_disaster.json  ->  0
pinery-bushfire_00001235_post_disaster.json  ->  0
woolsey-fire_00000404_post_disaster.json  ->  0
woolsey-fire_00000276_post_disaster.json  ->  0
woolsey-fire_00000442_post_disaster.json  ->  0
portugal-wildfire_00000759_post_disaster.json  ->  0
portugal-wildfire_00001123_post_disaster.json  ->  6
woolsey-fire_00000178_post_disaster.json  ->  1
pinery-bushfire_000015

## Verify Damage Classification Labels

This section locates a valid annotation file containing building polygons and inspects its metadata to confirm the presence of damage classification labels required for segmentation.

In [13]:
for idx in range(len(df)):

    row = df.iloc[idx]

    with open(row["label_json"], "r") as f:
        data = json.load(f)

    if len(data["features"]["xy"]) > 0:

        print(row["label_json"])
        print("Features:", len(data["features"]["xy"]))

        print("\nFirst feature properties:")
        print(data["features"]["xy"][0]["properties"])

        break

/kaggle/input/datasets/tunguz/xview2-challenge-dataset-tier-3-data/tier3/labels/joplin-tornado_00000000_post_disaster.json
Features: 185

First feature properties:
{'feature_type': 'building', 'subtype': 'destroyed', 'uid': 'b6aa615e-56b2-458a-b404-d094b64dcad3'}


## Quantify Empty Annotation Files

This section computes the proportion of annotation files that contain no building polygons, providing insight into dataset sparsity and potential class imbalance.

In [14]:
empty_count = 0

for path in df["label_json"]:

    with open(path, "r") as f:
        data = json.load(f)

    if len(data["features"]["xy"]) == 0:
        empty_count += 1

print("Empty:", empty_count)
print("Total:", len(df))
print("Percent:", 100 * empty_count / len(df))

Empty: 2991
Total: 6369
Percent: 46.96184644371173


## Train/test/validation split

In [15]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

Train: 4458
Val: 955
Test: 956


- We convert the dataset into CSV files to create a clean and permanent mapping between pre-image, post-image, and corresponding label JSON, making train/val/test splitting and loading faster, reproducible, and error-free.

- We use only post-disaster labels because damage severity classes (minor/major/destroyed) are defined only after the disaster, while pre-disaster labels (if present) mainly represent building footprints and do not contain damage information.

## Saving CSVs

In [16]:
train_df.to_csv("train_pairs.csv", index=False)
val_df.to_csv("val_pairs.csv", index=False)
test_df.to_csv("test_pairs.csv", index=False)

print("Saved: train_pairs.csv, val_pairs.csv, test_pairs.csv")

Saved: train_pairs.csv, val_pairs.csv, test_pairs.csv
